# CNN-BiLSTM-Attention + ECG Intervals — Tiffany Flat 3-class

Hybrid model: CNN-BiLSTM-Attention backbone + auxiliary ECG interval features.

**Interval features** (6-dim, extracted via NeuroKit2 from lead II):
- `rr_ms` — median RR interval (heart rate proxy)
- `rr_std_ms` — RR variability (AF proxy; MS median=99ms vs Control=22ms)
- `p_dur_ms` — P wave duration
- `qrs_dur_ms` — QRS duration
- `qtc_ms` — corrected QT interval (Bazett)
- `sokolow` — SV1+RV5 amplitude (LVH proxy)

**Architecture:**
```
ECG (12-ch) → MultiScale CNN (k=7/15/31) → proj(128) → BiLSTM(256) → Temporal Attention → 512-dim ─┐
                                                                                                      ├─ cat(544) → Dropout → Linear(3)
ECG intervals → norm → Linear(6→32) → ReLU → 32-dim ─────────────────────────────────────────────────┘
```

In [ ]:
from __future__ import annotations

import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
PROJECT_ROOT   = Path("/kaggle/input/datasets/pascalhafidzfajri/mlcv-mitral")
OUT_DIR        = Path(".")
NPZ_DIR        = PROJECT_ROOT / "mitral_images" / "mitral_images"
SPLIT_CSV_PATH = PROJECT_ROOT / "data_split_3level_exp10.csv"
INTERVALS_CSV  = PROJECT_ROOT / "ecg_intervals.csv"
ARTIFACT_DIR   = OUT_DIR / "dl_artifacts" / "cnn_bilstm_ecg_intervals_tiffany_flat"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES         = ["0_control", "1_significant_mr", "2_significant_ms"]
FLAT_LABEL_TO_CLASS = {"Kontrol": 0, "MR": 1, "MS": 2}
LEAD_ORDER          = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]

INTERVAL_COLS = ["rr_ms", "rr_std_ms", "p_dur_ms", "qrs_dur_ms", "qtc_ms", "sokolow"]
N_INTERVAL    = len(INTERVAL_COLS)  # 6

DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IN_CHANNELS    = len(LEAD_ORDER)  # 12
BATCH_SIZE     = 32
EPOCHS         = 60
PATIENCE       = 15
LEARNING_RATE  = 1e-3
MIN_LR         = 1e-5
WEIGHT_DECAY   = 1e-4
GRAD_CLIP_NORM = 5.0
NUM_WORKERS    = 0
PRELOAD_TO_RAM       = True
SKIP_COMPLETED_FOLDS = True

AUGMENTATION_CONFIG = {
    "noise_prob": 0.50, "noise_std": 0.01,
    "scale_prob": 0.50, "scale_range": (0.95, 1.05),
    "time_mask_prob": 0.35, "time_mask_max_frac": 0.03,
    "lead_dropout_prob": 0.15, "max_dropped_leads": 1,
}

print("Device:", DEVICE)
print("INTERVALS_CSV exists:", INTERVALS_CSV.exists())

## 1. Load and validate ECG intervals

In [ ]:
intervals_df = pd.read_csv(INTERVALS_CSV)
print(f"Intervals loaded: {len(intervals_df)} samples")
print(f"NaN rates: {intervals_df[INTERVAL_COLS].isna().mean().round(3).to_dict()}")

INTERVAL_LOOKUP = dict(
    zip(intervals_df["stem"].astype(str),
        intervals_df[INTERVAL_COLS].values.astype(np.float32))
)
print(f"Lookup built: {len(INTERVAL_LOOKUP)} stems")

## 2. Data loading

In [ ]:
NPZ_RAM_CACHE: dict = {}

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def infer_original_code(stem):
    stem = str(stem).strip().upper()
    for p in ("BA", "BB", "A", "B", "C", "D", "E"):
        if stem.startswith(p): return p
    return "UNKNOWN"

def normalize_per_sample_per_lead(signal):
    finite = np.isfinite(signal)
    vc  = np.maximum(finite.sum(axis=0, keepdims=True).astype(np.float32), 1.0)
    mu  = np.where(finite, signal, 0.0).sum(axis=0, keepdims=True).astype(np.float32) / vc
    c   = np.where(finite, signal - mu, 0.0).astype(np.float32)
    std = np.sqrt(np.maximum((c**2).sum(axis=0, keepdims=True) / vc, 1e-6)).astype(np.float32)
    std = np.where(finite.sum(axis=0, keepdims=True) == 0, 1.0, std)
    mu  = np.where(finite.sum(axis=0, keepdims=True) == 0, 0.0, mu)
    return np.nan_to_num((np.where(finite, signal, mu) - mu) / std,
                         nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def augment_signal_channels(signal_ch, rng, cfg):
    x = signal_ch.copy(); nl, ns = x.shape
    if rng.random() < cfg["scale_prob"]:
        lo, hi = cfg["scale_range"]; x *= rng.uniform(lo, hi, size=(nl, 1)).astype(np.float32)
    if rng.random() < cfg["noise_prob"] and cfg["noise_std"] > 0:
        x += rng.normal(0.0, cfg["noise_std"], size=x.shape).astype(np.float32)
    if rng.random() < cfg["time_mask_prob"] and cfg["time_mask_max_frac"] > 0:
        mw = max(1, int(ns * cfg["time_mask_max_frac"]))
        w  = int(rng.integers(1, mw + 1))
        s  = int(rng.integers(0, max(1, ns - w + 1)))
        x[:, s:s + w] = 0.0
    if rng.random() < cfg["lead_dropout_prob"] and cfg["max_dropped_leads"] > 0:
        nd = int(rng.integers(1, cfg["max_dropped_leads"] + 1))
        x[rng.choice(nl, size=nd, replace=False), :] = 0.0
    return x.astype(np.float32)

def load_npz_arrays(npz_path):
    with np.load(npz_path, allow_pickle=False) as d:
        return (d["signal_matrix"].astype(np.float32),
                d["overlap_mask"].astype(np.uint8),
                d["interpolated_mask"].astype(np.uint8))

def get_npz_arrays(row):
    key = str(row["stem"])
    if not PRELOAD_TO_RAM: return load_npz_arrays(row["npz_path"])
    if key not in NPZ_RAM_CACHE: NPZ_RAM_CACHE[key] = load_npz_arrays(row["npz_path"])
    return NPZ_RAM_CACHE[key]

def preload_npz_frame(frame, desc):
    if not PRELOAD_TO_RAM: return
    pending = [(str(r["stem"]), str(r["npz_path"]))
               for _, r in frame.drop_duplicates("stem").iterrows()
               if str(r["stem"]) not in NPZ_RAM_CACHE]
    if not pending: return
    def load_one(item): return item[0], load_npz_arrays(item[1])
    with ThreadPoolExecutor(max_workers=4) as ex:
        for stem, arrays in tqdm(
                (f.result() for f in as_completed(ex.submit(load_one, i) for i in pending)),
                total=len(pending), desc=f"preload {desc}", leave=False):
            NPZ_RAM_CACHE[stem] = arrays


class MitralECGDataset(Dataset):
    """Returns (signal_12ch, interval_vec, label, stem)."""
    def __init__(self, frame, interval_means, interval_stds, augment=False, seed=SEED):
        self.frame  = frame.reset_index(drop=True)
        self.means  = interval_means
        self.stds   = interval_stds
        self.augment = augment
        self.seed   = seed
        preload_npz_frame(self.frame, "ECG")

    def __len__(self): return len(self.frame)

    def __getitem__(self, idx):
        row  = self.frame.iloc[idx]
        stem = str(row["stem"])

        signal, overlap, _ = get_npz_arrays(row)
        signal[overlap.astype(bool)] = 0.0
        signal_ch = normalize_per_sample_per_lead(signal).T  # (12, T)
        if self.augment:
            rng = np.random.default_rng(self.seed + idx * 1009 + random.randint(0, 1_000_000))
            signal_ch = augment_signal_channels(signal_ch, rng, AUGMENTATION_CONFIG)

        raw  = INTERVAL_LOOKUP.get(stem, np.full(N_INTERVAL, np.nan, dtype=np.float32))
        raw  = np.where(np.isnan(raw), self.means, raw)
        ivec = ((raw - self.means) / np.maximum(self.stds, 1e-6)).astype(np.float32)

        return (
            torch.from_numpy(signal_ch.astype(np.float32)),
            torch.from_numpy(ivec),
            torch.tensor(int(row["class_id"]), dtype=torch.long),
            stem,
        )


def compute_interval_stats(frame):
    stems = frame["stem"].astype(str).tolist()
    vecs  = [INTERVAL_LOOKUP[s] for s in stems if s in INTERVAL_LOOKUP]
    arr   = np.array(vecs, dtype=np.float32)
    means = np.nanmedian(arr, axis=0)
    stds  = np.maximum(np.nanstd(arr, axis=0), 1e-6)
    return means.astype(np.float32), stds.astype(np.float32)


def load_tiffany_split():
    split = pd.read_csv(SPLIT_CSV_PATH)
    split["stem"]     = split["filename"].map(lambda x: Path(str(x)).stem)
    split["original_code"] = split["stem"].map(infer_original_code)
    split["npz_path"] = split["stem"].map(lambda s: str(NPZ_DIR / f"{s}.npz"))
    split["npz_exists"] = split["npz_path"].map(lambda p: Path(p).exists())
    avail = split[split["npz_exists"]].copy().reset_index(drop=True)
    avail["class_id"] = avail["flat_label"].map(FLAT_LABEL_TO_CLASS)
    print(f"Available: {len(avail)}/{len(split)}")
    tv  = avail[avail["isHeldOut"] == 0].copy().reset_index(drop=True)
    hld = avail[avail["isHeldOut"] == 1].copy().reset_index(drop=True)
    print(f"Train/val: {len(tv)}, Heldout: {len(hld)}")
    return tv, hld


def make_loader(frame, means, stds, batch_size, shuffle, augment, seed):
    ds = MitralECGDataset(frame, means, stds, augment=augment, seed=seed)
    g  = torch.Generator().manual_seed(seed)
    return DataLoader(ds, batch_size, shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=(DEVICE.type == "cuda"), generator=g)


set_seed(SEED)
train_val_df, heldout_df = load_tiffany_split()

## 3. CNN-BiLSTM-Attention + Intervals architecture

```
Input: (B, 12, T)
  ├─ Branch-S: Conv(k=7) × 2 → MaxPool → 32-ch
  ├─ Branch-M: Conv(k=15) × 2 → MaxPool → 32-ch
  └─ Branch-L: Conv(k=31) × 2 → MaxPool → 32-ch
  → cat(96-ch) → Conv1x1 proj → 128-ch
  → BiLSTM(256, bidirectional) → (B, T', 512)
  → Temporal soft attention → 512-dim
  → cat with interval branch (32-dim) → 544-dim
  → Dropout(0.3) → Linear(544, 3)
```

In [ ]:
class CNNBranch(nn.Module):
    def __init__(self, in_ch, out_ch, kernel, pool=4):
        super().__init__()
        p = kernel // 2
        self.net = nn.Sequential(
            nn.Conv1d(in_ch,  out_ch, kernel, padding=p, bias=False),
            nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True),
            nn.Conv1d(out_ch, out_ch, kernel, padding=p, bias=False),
            nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True),
            nn.MaxPool1d(pool),
        )
    def forward(self, x): return self.net(x)


class TemporalAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.score = nn.Linear(hidden_size, 1)
    def forward(self, x):  # x: (B, T, H)
        w = torch.softmax(self.score(x), dim=1)  # (B, T, 1)
        return (x * w).sum(dim=1)                # (B, H)


class CNNBiLSTMWithIntervals(nn.Module):
    """
    CNN-BiLSTM-Attention backbone fused with ECG interval features.
    Backbone attention output (512-dim) + interval branch (32-dim) → head (3-class).
    """
    def __init__(self, in_channels=12, num_classes=3,
                 branch_ch=32, lstm_hidden=256,
                 n_interval=6, interval_hidden=32, dropout=0.3):
        super().__init__()
        # Multi-scale CNN branches
        self.branch_s = CNNBranch(in_channels, branch_ch, kernel=7)
        self.branch_m = CNNBranch(in_channels, branch_ch, kernel=15)
        self.branch_l = CNNBranch(in_channels, branch_ch, kernel=31)

        feat_ch = branch_ch * 3  # 96
        self.proj = nn.Sequential(
            nn.Conv1d(feat_ch, 128, 1, bias=False),
            nn.BatchNorm1d(128), nn.ReLU(inplace=True),
        )
        self.bilstm = nn.LSTM(
            input_size=128, hidden_size=lstm_hidden,
            num_layers=1, batch_first=True,
            bidirectional=True, dropout=0.0,
        )
        self.attention = TemporalAttention(lstm_hidden * 2)  # 512-dim

        # Interval branch
        self.interval_branch = nn.Sequential(
            nn.Linear(n_interval, interval_hidden),
            nn.ReLU(inplace=True),
        )

        # Fusion head: 512 + 32 = 544
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(lstm_hidden * 2 + interval_hidden, num_classes)

    def forward(self, x, ivec):  # x: (B, 12, T), ivec: (B, 6)
        f_s = self.branch_s(x)
        f_m = self.branch_m(x)
        f_l = self.branch_l(x)
        L    = min(f_s.shape[-1], f_m.shape[-1], f_l.shape[-1])
        feat = torch.cat([f_s[..., :L], f_m[..., :L], f_l[..., :L]], dim=1)  # (B, 96, L)
        feat = self.proj(feat).transpose(1, 2)   # (B, L, 128)
        out, _ = self.bilstm(feat)               # (B, L, 512)
        backbone_feat = self.attention(out)       # (B, 512)
        interval_feat = self.interval_branch(ivec)  # (B, 32)
        fused = torch.cat([backbone_feat, interval_feat], dim=1)  # (B, 544)
        return self.head(self.drop(fused))


_m = CNNBiLSTMWithIntervals(IN_CHANNELS, len(CLASS_NAMES))
print(f"CNNBiLSTMWithIntervals: {sum(p.numel() for p in _m.parameters()):,} parameters")
del _m

## 4. Training utilities

In [ ]:
def stable_softmax(logits):
    x = np.asarray(logits, dtype=np.float64)
    x -= x.max(axis=1, keepdims=True)
    e  = np.exp(x)
    return (e / e.sum(axis=1, keepdims=True)).astype(np.float32)


def metrics_from_probs(y_true, probs, class_names, prefix=""):
    pred    = probs.argmax(axis=1)
    labels  = list(range(len(class_names)))
    recalls = recall_score(y_true, pred, labels=labels, average=None, zero_division=0)
    return {
        f"{prefix}accuracy":          accuracy_score(y_true, pred),
        f"{prefix}balanced_accuracy": balanced_accuracy_score(y_true, pred),
        f"{prefix}macro_f1":          f1_score(y_true, pred, average="macro", zero_division=0),
        f"{prefix}weighted_f1":       f1_score(y_true, pred, average="weighted", zero_division=0),
        f"{prefix}macro_precision":   precision_score(y_true, pred, average="macro", zero_division=0),
        f"{prefix}macro_recall":      recall_score(y_true, pred, average="macro", zero_division=0),
        **{f"{prefix}recall_{c}": float(recalls[i]) for i, c in enumerate(class_names)},
        f"{prefix}min_class_recall":  float(recalls.min()),
        f"{prefix}macro_ovr_auc":     roc_auc_score(y_true, probs, multi_class="ovr", average="macro"),
    }


def class_weighted_ce(frame, label_col, n_classes):
    w = compute_class_weight("balanced", classes=np.arange(n_classes), y=frame[label_col].values)
    return nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32).to(DEVICE))


def run_epoch(model, loader, criterion, optimizer, scaler, training, desc):
    model.train(training)
    total_loss, all_logits, all_labels, all_stems = 0.0, [], [], []
    ctx = torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda"))
    for x, ivec, y, stems in tqdm(loader, desc=desc, leave=False):
        x, ivec, y = x.to(DEVICE), ivec.to(DEVICE), y.to(DEVICE)
        with ctx:
            logits = model(x, ivec)
            loss   = criterion(logits, y)
        if training:
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
        total_loss += loss.item() * len(y)
        all_logits.append(logits.detach().cpu().float().numpy())
        all_labels.append(y.cpu().numpy())
        all_stems.extend(stems)
    return (
        total_loss / len(loader.dataset),
        np.concatenate(all_labels),
        np.concatenate(all_logits),
        np.array(all_stems),
    )

## 5. 5-fold cross-validation

In [ ]:
def train_one_fold(fold, train_df, val_df):
    fold_dir     = ARTIFACT_DIR / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    best_path    = fold_dir / "best_model.pt"
    metrics_path = fold_dir / "heldout_metrics.csv"
    logits_path  = fold_dir / "heldout_logits.npz"

    if SKIP_COMPLETED_FOLDS and metrics_path.exists() and logits_path.exists() and best_path.exists():
        print(f"Fold {fold}: done — skipping.")
        return pd.read_csv(metrics_path).iloc[0].to_dict()

    seed = SEED + fold
    set_seed(seed)

    means, stds = compute_interval_stats(train_df)
    np.save(fold_dir / "interval_stats.npy", np.stack([means, stds]))

    train_loader = make_loader(train_df,   means, stds, BATCH_SIZE, True,  True,  seed)
    val_loader   = make_loader(val_df,     means, stds, BATCH_SIZE, False, False, seed)
    held_loader  = make_loader(heldout_df, means, stds, BATCH_SIZE, False, False, seed)

    model     = CNNBiLSTMWithIntervals(IN_CHANNELS, len(CLASS_NAMES)).to(DEVICE)
    criterion = class_weighted_ce(train_df, "class_id", len(CLASS_NAMES))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LR)
    scaler    = torch.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    best_score, patience_ctr = -np.inf, 0
    history = []

    for epoch in tqdm(range(1, EPOCHS + 1), desc=f"fold {fold}"):
        tl, ty, tlo, _ = run_epoch(model, train_loader, criterion, optimizer, scaler, True,  f"f{fold} tr")
        vl, vy, vlo, _ = run_epoch(model, val_loader,   criterion, None,      None,   False, f"f{fold} val")
        scheduler.step()
        tp = stable_softmax(tlo); vp = stable_softmax(vlo)
        tm = metrics_from_probs(ty, tp, CLASS_NAMES, "train_")
        vm = metrics_from_probs(vy, vp, CLASS_NAMES, "val_")
        score = vm["val_macro_f1"]
        history.append({"fold": fold, "epoch": epoch,
                        "train_loss": tl, "val_loss": vl,
                        "best_score": score, **tm, **vm})
        tqdm.write(
            f"fold={fold} ep={epoch:03d} tloss={tl:.4f} vloss={vl:.4f} "
            f"vf1={vm['val_macro_f1']:.4f} vauc={vm['val_macro_ovr_auc']:.4f}"
        )
        if score > best_score:
            best_score = score; patience_ctr = 0
            torch.save({"model_state_dict": model.state_dict(), "fold": fold,
                        "epoch": epoch, "best_score": best_score,
                        "interval_means": means, "interval_stds": stds}, best_path)
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                tqdm.write(f"fold={fold} early stop at epoch {epoch}"); break

    pd.DataFrame(history).to_csv(fold_dir / "training_history.csv", index=False)

    ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    best_epoch = ckpt["epoch"]

    _, hy, hl, hs = run_epoch(model, held_loader, criterion, None, None, False, f"f{fold} held")
    hp = stable_softmax(hl)
    hm = metrics_from_probs(hy, hp, CLASS_NAMES, "heldout_")
    hm.update({"fold": fold, "best_epoch": best_epoch, "best_score": best_score})
    pd.DataFrame([hm]).to_csv(metrics_path, index=False)
    np.savez(logits_path, y_true=hy, probs=hp, stems=hs)
    train_df.to_csv(fold_dir / "train_split.csv", index=False)
    val_df.to_csv(fold_dir   / "val_split.csv",   index=False)

    print(
        f"Fold {fold}: F1={hm['heldout_macro_f1']:.4f} Bal={hm['heldout_balanced_accuracy']:.4f} "
        f"AUC={hm['heldout_macro_ovr_auc']:.4f} "
        f"Ctrl={hm['heldout_recall_0_control']:.3f} "
        f"MR={hm['heldout_recall_1_significant_mr']:.3f} "
        f"MS={hm['heldout_recall_2_significant_ms']:.3f} ep={best_epoch}"
    )
    return hm

In [ ]:
set_seed(SEED)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_val_df, train_val_df["class_id"])):
    train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
    val_df   = train_val_df.iloc[val_idx].reset_index(drop=True)
    result   = train_one_fold(fold, train_df, val_df)
    fold_results.append(result)

fold_df = pd.DataFrame(fold_results)
fold_df.to_csv(ARTIFACT_DIR / "heldout_fold_metrics.csv", index=False)
print("\n=== Per-fold heldout metrics ===")
print(fold_df[[
    "fold", "heldout_macro_f1", "heldout_balanced_accuracy", "heldout_macro_ovr_auc",
    "heldout_recall_0_control", "heldout_recall_1_significant_mr",
    "heldout_recall_2_significant_ms", "best_epoch",
]].to_string(index=False))

## 6. Ensemble evaluation

In [ ]:
probs_list, y_ref, stems_ref = [], None, None
for fold in range(5):
    d = np.load(ARTIFACT_DIR / f"fold_{fold}/heldout_logits.npz", allow_pickle=False)
    if y_ref is None: y_ref, stems_ref = d["y_true"], d["stems"]
    probs_list.append(d["probs"])

ens_probs = np.mean(probs_list, axis=0)
ens_pred  = ens_probs.argmax(axis=1)
ens_m     = metrics_from_probs(y_ref, ens_probs, CLASS_NAMES, "heldout_")
ens_m["n_models"] = 5
pd.DataFrame([ens_m]).to_csv(ARTIFACT_DIR / "heldout_ensemble_metrics.csv", index=False)

print("=== CNN-BiLSTM+Intervals 5-fold Soft Ensemble ===")
print(f"  F1={ens_m['heldout_macro_f1']:.4f}  Bal={ens_m['heldout_balanced_accuracy']:.4f}  AUC={ens_m['heldout_macro_ovr_auc']:.4f}")
print(f"  Ctrl={ens_m['heldout_recall_0_control']:.3f}  MR={ens_m['heldout_recall_1_significant_mr']:.3f}  MS={ens_m['heldout_recall_2_significant_ms']:.3f}")
print()
print(classification_report(y_ref, ens_pred, target_names=CLASS_NAMES, zero_division=0))

cm      = confusion_matrix(y_ref, ens_pred, labels=[0, 1, 2])
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
pd.DataFrame(cm,      index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(ARTIFACT_DIR / "heldout_confusion_matrix.csv")
pd.DataFrame(cm_norm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(ARTIFACT_DIR / "heldout_confusion_matrix_normalized.csv")